# DiTTO — Text-Native Product-Pair Classification

Fine-tunes a pretrained transformer (RoBERTa) on the multimodal-KG notebook's
candidate pairs, following Li et al., *"Deep Entity Matching with Pre-Trained
Language Models"* (DiTTO, VLDB 2020).

DiTTO's recipe: serialize each product as `COL <attr> VAL <value>` tokens,
concatenate the two products with a `[SEP]`, and let the LM's `[CLS]` vector
drive a 3-class head (**duplicate / variant / unrelated**).

### Files this notebook reads

Everything is written by `multimodal_kg_new_FIXED_.ipynb`:

| File | Used for |
|---|---|
| `candidate_feature_table_multimodal.parquet` | candidate pairs |
| `product_knowledge_graph.parquet` | recovers brand + leaf_category per item |
| `item_image_urls.parquet` | supplies the product row order |

### Two fixes handled automatically in Cell 3

1. **`df_products.parquet` is not saved by the KG notebook** — Cell 3 builds it
   from the two KG files above.
2. **`weak_label` is missing from the candidate file** — the KG notebook saves
   the candidate file at cell 91 but only adds `weak_label` at cell 95 and
   never re-saves. Cell 3 recomputes the labels using the KG notebook's exact
   rules.

### Outputs

In `OUT_DIR`: `best_model.pt`, `label_encoder.json`, `metrics.json`,
`test_predictions.parquet`.

> **Set `DRY_RUN = True` for a fast CPU-only pipeline check.** Flip to `False`
> and Restart & Run All for the real fine-tuning run on GPU.

## 1. Setup

Install dependencies once, then import everything.

In [ ]:
# Run once if these aren't already installed:
# %pip install torch transformers scikit-learn pandas pyarrow numpy

In [ ]:
import json, os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))

### Configuration

Set the paths so they point at the directory that has your KG artifacts.

In [ ]:
# ---- paths (edit to match your KG notebook's output directory) ----
DATA_DIR      = "."                                 # where the KG artifacts live
PAIRS_PATH    = f"{DATA_DIR}/candidate_feature_table_multimodal.parquet"
KG_PATH       = f"{DATA_DIR}/product_knowledge_graph.parquet"
IMG_URLS_PATH = f"{DATA_DIR}/item_image_urls.parquet"
PRODUCTS_PATH = f"{DATA_DIR}/df_products.parquet"   # auto-built in Cell 3

OUT_DIR       = "runs/ditto"

# ---- model / training ----
MODEL_NAME  = "roberta-base"    # DiTTO default; try bert-base-uncased instead
EPOCHS      = 3
BATCH_SIZE  = 32
LR          = 2e-5
MAX_LEN     = 128
MAX_PAIRS   = None              # e.g. 5000 for a fast smoke test
SEED        = 42
NUM_WORKERS = 2

# ---- set True to test plumbing on CPU with no model download ----
DRY_RUN     = True

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

## 2. File check + auto-repair

Run this cell first. It:

1. Lists which files exist and which are missing.
2. If `df_products.parquet` is missing, rebuilds it from
   `item_image_urls.parquet` + `product_knowledge_graph.parquet`.
3. If the candidate table has no `weak_label` column (this is the actual state
   after running the KG notebook — the file is saved *before* `weak_label` is
   added, and never re-saved), adds `spec_diff`, `weak_label`, and `confidence`
   using the KG notebook's exact rules.

After this cell, everything downstream is drop-in — same code as the working
pipeline.

In [ ]:
# ==================================================================
# Auto-repair helpers — self-contained (paste-verbatim into other nbs)
# ==================================================================
import re

def rebuild_df_products(out_path, item_image_urls_path, kg_path):
    """Rebuild df_products.parquet from KG artifacts (df was never saved).

    Row order = row order of item_image_urls.parquet, matching the order the KG
    notebook keeps df in when it builds candidates. This makes item_i / item_j
    from the candidate table align with product rows.

    Brand and leaf_category are recovered from product_knowledge_graph.parquet
    via has_brand and belongs_to edges. Titles aren't in any KG artifact, so
    they are placeholders (`item_<asin>`); the models still train.
    """
    print(f"[rebuild] building {out_path} from KG artifacts...")
    urls = pd.read_parquet(item_image_urls_path)
    if "item_id" not in urls.columns:
        raise ValueError(f"{item_image_urls_path} missing 'item_id' column.")

    kg = pd.read_parquet(kg_path)
    if not {"head", "relation", "tail"}.issubset(kg.columns):
        raise ValueError(f"{kg_path} missing head/relation/tail columns.")

    brand_map, cat_map = {}, {}
    for _, r in kg[kg["relation"] == "has_brand"].iterrows():
        head = str(r["head"]); tail = str(r["tail"])
        if not head.startswith("item_"):
            continue
        asin = head[5:]
        brand_map[asin] = tail[6:] if tail.startswith("brand_") else tail

    for _, r in kg[kg["relation"] == "belongs_to"].iterrows():
        head = str(r["head"]); tail = str(r["tail"])
        if not head.startswith("item_"):
            continue           # skip taxonomy->taxonomy edges
        asin = head[5:]
        val  = tail[9:] if tail.startswith("category_") else tail
        cat_map.setdefault(asin, val)   # first belongs_to is the leaf

    ids = urls["item_id"].astype(str).values
    prod = pd.DataFrame({
        "item_id":       ids,
        "title":         [f"item_{a}" for a in ids],
        "brand":         [brand_map.get(a) for a in ids],
        "leaf_category": [cat_map.get(a)   for a in ids],
    })
    prod.to_parquet(out_path, index=False)

    b, c = prod["brand"].notna().mean(), prod["leaf_category"].notna().mean()
    print(f"[rebuild] wrote {out_path}: {len(prod):,} rows | "
          f"brand coverage {b:.1%} | leaf_category coverage {c:.1%}")
    print("[rebuild] NOTE: titles are placeholders (df.title was never saved).")
    print("[rebuild] For real titles, add this cell to your KG notebook after df")
    print("[rebuild] is built (before candidates are built, do NOT re-sort df):")
    print("[rebuild]   df[['item_id','title','brand','leaf_category']] \\")
    print("[rebuild]     .reset_index(drop=True).to_parquet('df_products.parquet',")
    print("[rebuild]                                        index=False)")
    return prod


# ---- weak_label rules (ported verbatim from KG cells 92 + 94) ----
SPEC_WORDS = {
    "gb","tb","mb","mah","hz","ghz","mhz","inch","cm","mm","ft",
    "oz","ml","l","kg","g","lb","pack","packs","ct","count","pc",
    "pcs","piece","pieces","yr","year","years",
}
_NUM_UNIT_RE = re.compile(r"(\d+(?:\.\d+)?)([a-z]+)")

def _extract_specs(title):
    if not isinstance(title, str):
        return set()
    t, out = title.lower(), set()
    for num, unit in _NUM_UNIT_RE.findall(t):
        if unit in SPEC_WORDS:
            out.add(f"{num}{unit}")
    for w in t.split():
        if w in SPEC_WORDS:
            out.add(w)
    return out

def spec_difference(title_i, title_j):
    return bool(_extract_specs(title_i).symmetric_difference(_extract_specs(title_j)))


def _weak_label_row(row):
    """KG cell 94, verbatim."""
    if (row.similarity > 0.92
        and row.image_similarity > 0.90
        and row.brand_match == 1):
        if row.spec_diff:
            return "variant", 0.85
        return "duplicate", 0.95
    if (row.similarity > 0.80
        and row.brand_match == 1
        and row.category_match == 1):
        return "variant", 0.80
    if row.similarity < 0.45 and row.brand_match == 0:
        return "unrelated", 0.90
    return "unknown", 0.5


def add_weak_labels(candidates, products):
    """Add spec_diff + weak_label + confidence in-place if missing. Uses the KG
    notebook's exact rules; produces the same labels the KG notebook would
    have produced if it had re-saved the file at the end."""
    added = []
    if "spec_diff" not in candidates.columns:
        titles = products["title"].astype(str).values
        candidates["spec_diff"] = candidates.apply(
            lambda r: spec_difference(titles[int(r.item_i)],
                                      titles[int(r.item_j)]),
            axis=1,
        )
        added.append("spec_diff")

    if "weak_label" not in candidates.columns or "confidence" not in candidates.columns:
        for col in ["similarity","image_similarity","brand_match",
                    "category_match","spec_diff"]:
            if col not in candidates.columns:
                raise ValueError(
                    f"cannot compute weak_label: candidates missing '{col}'. "
                    f"Present columns: {list(candidates.columns)}")
        lc = candidates.apply(_weak_label_row, axis=1, result_type="expand")
        candidates["weak_label"] = lc[0]
        candidates["confidence"] = lc[1]
        added += ["weak_label","confidence"]

    if added:
        print(f"[repair] added columns to candidates: {added}")
        print("[repair] weak_label distribution:")
        print(candidates["weak_label"].value_counts().to_string())
    else:
        print("[repair] candidates already has weak_label — nothing to do.")
    return candidates

In [ ]:
# =====================  FILE CHECK  =====================
_required_paths = [
    ("PAIRS_PATH", PAIRS_PATH, "candidate pairs from KG notebook"),
    ("KG_PATH",    KG_PATH,    "product KG (rebuilds df_products; supplies brand/leaf_category)"),
    ("IMG_URLS_PATH", IMG_URLS_PATH, "list of item_ids (defines product row order)"),
]

_optional_paths = [
    ("PRODUCTS_PATH", PRODUCTS_PATH, "auto-built below if missing"),
]

for name, path, desc in _required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"required file '{path}' (variable {name}) not found. "
            f"Purpose: {desc}. Check that PAIRS_PATH / KG_PATH / IMG_URLS_PATH "
            "in the config cell point at your KG notebook's output directory.")
    print(f"[check]  OK      {name:15s} {path}  ({os.path.getsize(path)/1e6:.1f} MB)")

for name, path, desc in _optional_paths:
    if os.path.exists(path):
        print(f"[check]  OK      {name:15s} {path}  ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        print(f"[check]  MISSING {name:15s} {path}  ({desc})")

# =====================  AUTO-BUILD df_products.parquet  =====================
if not os.path.exists(PRODUCTS_PATH):
    rebuild_df_products(PRODUCTS_PATH, IMG_URLS_PATH, KG_PATH)
else:
    print(f"[rebuild] {PRODUCTS_PATH} already present — skipping rebuild.")

# =====================  AUTO-ADD weak_label to candidates  =====================
# We ALWAYS check on load (rather than write a new file) so re-runs on a
# freshly-regenerated candidate table just work.
_prod_for_repair = pd.read_parquet(PRODUCTS_PATH)
_cand_for_repair = pd.read_parquet(PAIRS_PATH)
_needs_labels = ("weak_label" not in _cand_for_repair.columns
                 or "confidence" not in _cand_for_repair.columns)
if _needs_labels:
    _cand_for_repair = add_weak_labels(_cand_for_repair, _prod_for_repair)
    # write to a companion file so downstream cells load the version with labels
    LABELED_PAIRS_PATH = PAIRS_PATH.replace(".parquet", "_labeled.parquet")
    _cand_for_repair.to_parquet(LABELED_PAIRS_PATH, index=False)
    print(f"[repair] wrote {LABELED_PAIRS_PATH} — downstream cells will use it.")
    PAIRS_PATH = LABELED_PAIRS_PATH
else:
    print("[repair] candidates already has weak_label + confidence.")

## 3. Serialization

Turn each product into `COL <attr> VAL <value>` text — attribute boundaries
stay explicit to the language model.

In [ ]:
SERIALIZE_FIELDS = ["title", "brand", "leaf_category"]

def serialize_entity(row: pd.Series) -> str:
    parts = []
    for col in SERIALIZE_FIELDS:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            parts.append(f"COL {col} VAL {str(row[col]).strip()}")
    return " ".join(parts) if parts else "COL title VAL [unknown]" 

## 4. Load and prepare

Filters to the 3 target classes, bounds-checks pair indices, encodes labels.

In [ ]:
KEEP_LABELS = ["duplicate", "variant", "unrelated"]

def load_and_prepare(pairs_path, products_path, seed, max_pairs=None):
    pairs    = pd.read_parquet(pairs_path)
    products = pd.read_parquet(products_path)

    if "weak_label" not in pairs.columns:
        raise ValueError(
            f"'weak_label' still missing in {pairs_path} — did Cell 3 run?")

    n_before = len(pairs)
    pairs = pairs[pairs["weak_label"].isin(KEEP_LABELS)].reset_index(drop=True)
    print(f"[data] kept {len(pairs):,}/{n_before:,} pairs "
          f"(dropped 'unknown' and any other non-target labels).")

    if len(pairs) == 0:
        raise ValueError(
            "0 pairs left after filtering — every candidate got labeled "
            "'unknown'. Loosen the KG notebook's weak_label thresholds or "
            "add more similarity/brand-match signal.")

    n_prod = len(products)
    bad = pairs[(pairs["item_i"] >= n_prod) | (pairs["item_j"] >= n_prod) |
                (pairs["item_i"] < 0) | (pairs["item_j"] < 0)]
    if len(bad):
        raise ValueError(
            f"{len(bad)} pairs reference row index outside [0,{n_prod}). "
            "Check that df_products.parquet was built from the same "
            "item_image_urls.parquet the candidates were built against.")

    if max_pairs and len(pairs) > max_pairs:
        pairs = pairs.sample(max_pairs, random_state=seed).reset_index(drop=True)
        print(f"[data] subsampled to {max_pairs:,} pairs (MAX_PAIRS).")

    label2id = {lbl: k for k, lbl in enumerate(KEEP_LABELS)}
    id2label = {k: lbl for lbl, k in label2id.items()}
    pairs["label_id"] = pairs["weak_label"].map(label2id).astype(int)
    if "confidence" not in pairs.columns:
        pairs["confidence"] = 1.0

    print("[data] class distribution:")
    print(pairs["weak_label"].value_counts().to_string())
    return pairs, products, label2id, id2label

pairs, products, label2id, id2label = load_and_prepare(
    PAIRS_PATH, PRODUCTS_PATH, SEED, MAX_PAIRS)
num_classes = len(label2id)

## 5. Stratified train / val / test split (same seed as MKGformer).

In [ ]:
from sklearn.model_selection import train_test_split

def stratified_split(pairs, seed, val_frac=0.15, test_frac=0.15):
    idx = np.arange(len(pairs)); y = pairs["label_id"].values
    # sklearn stratify needs >= 2 members per class per fold; if any class is
    # rarer than that, fall back to an unstratified split and warn.
    counts = np.bincount(y)
    if counts.min() < 2:
        rare = np.where(counts < 2)[0]
        print(f"[split] WARN: class(es) {rare.tolist()} have < 2 members "
              f"({counts.tolist()}). Falling back to UNSTRATIFIED split — try "
              "loosening the KG notebook's weak_label thresholds to get more "
              "of the rare classes.")
        stratify_arg = None
    else:
        stratify_arg = y

    tr, tmp = train_test_split(idx, test_size=val_frac + test_frac,
                               stratify=stratify_arg, random_state=seed)
    rel = test_frac / (val_frac + test_frac)
    tmp_y = y[tmp] if stratify_arg is not None else None
    # re-check for the second split; the tmp subset might be even sparser
    if tmp_y is not None and np.bincount(tmp_y, minlength=len(counts)).min() < 2:
        tmp_y = None
        print("[split] WARN: rare class(es) in val+test subset — second split "
              "is unstratified.")
    va, te = train_test_split(tmp, test_size=rel, stratify=tmp_y,
                              random_state=seed)
    print(f"[split] train={len(tr):,}  val={len(va):,}  test={len(te):,}")
    return tr, va, te

train_idx, val_idx, test_idx = stratified_split(pairs, SEED)

## 6. Torch `Dataset`

In [ ]:
def make_dataset(pairs, products, indices, tokenizer, max_len):
    class PairDS(Dataset):
        def __init__(self):
            self.rows = pairs.iloc[indices].reset_index(drop=True)
        def __len__(self):
            return len(self.rows)
        def __getitem__(self, k):
            r = self.rows.iloc[k]
            left  = serialize_entity(products.iloc[int(r.item_i)])
            right = serialize_entity(products.iloc[int(r.item_j)])
            enc = tokenizer(left, right, truncation=True, max_length=max_len,
                            padding="max_length", return_tensors="pt")
            return {
                "input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "label":  torch.tensor(int(r.label_id), dtype=torch.long),
                "weight": torch.tensor(float(r.confidence), dtype=torch.float),
            }
    return PairDS()

## 7. Model and tokenizer

**Real run:** loads `AutoTokenizer` and `AutoModelForSequenceClassification`.
**Dry run:** a char-hash tokenizer + mean-pooling stub — offline / CPU / fast.

In [ ]:
class _StubTokenizer:
    def __init__(self, vocab=1000): self.vocab = vocab
    def __call__(self, a, b=None, truncation=True, max_length=64,
                 padding="max_length", return_tensors="pt"):
        text = a if b is None else (a + " [SEP] " + b)
        toks = [(hash(t) % (self.vocab - 1)) + 1 for t in text.split()][:max_length]
        ids  = toks + [0] * (max_length - len(toks))
        mask = [1]  * len(toks) + [0] * (max_length - len(toks))
        return {"input_ids": torch.tensor([ids]),
                "attention_mask": torch.tensor([mask])}

def _build_stub_model(num_classes, vocab=1000, dim=32):
    class Stub(nn.Module):
        def __init__(self):
            super().__init__()
            self.emb  = nn.Embedding(vocab, dim, padding_idx=0)
            self.head = nn.Linear(dim, num_classes)
        def forward(self, input_ids, attention_mask):
            e = self.emb(input_ids)
            m = attention_mask.unsqueeze(-1).float()
            p = (e * m).sum(1) / m.sum(1).clamp(min=1)
            return type("O", (), {"logits": self.head(p)})()
    return Stub()

In [ ]:
if DRY_RUN:
    tokenizer = _StubTokenizer()
    model = _build_stub_model(num_classes).to(device)
    print("[model] DRY RUN — stub encoder on", device)
else:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_classes,
        id2label=id2label, label2id=label2id).to(device)
    print(f"[model] {MODEL_NAME} loaded on {device}")

## 8. DataLoaders, class weights, optimizer

Loss = per-pair confidence × inverse-frequency-class-weighted cross-entropy.

In [ ]:
nw = 0 if DRY_RUN else NUM_WORKERS
def dl(idx, shuffle):
    return DataLoader(make_dataset(pairs, products, idx, tokenizer, MAX_LEN),
                      batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=nw,
                      pin_memory=(device.type == "cuda"))

train_loader = dl(train_idx, True)
val_loader   = dl(val_idx,   False)
test_loader  = dl(test_idx,  False)

def class_weights(pairs, indices, num_classes):
    y = pairs["label_id"].values[indices]
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    return torch.tensor(counts.sum() / (num_classes * counts), dtype=torch.float)

cw = class_weights(pairs, train_idx, num_classes)
print("[loss] class weights:", cw.tolist())

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

## 9. Train / evaluate

In [ ]:
def run_epoch(model, loader, device, optimizer=None, class_w=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, all_pred, all_true = 0.0, [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y    = batch["label"].to(device)
            w    = batch["weight"].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            logits = out.logits if hasattr(out, "logits") else out
            ce = F.cross_entropy(logits, y,
                                 weight=class_w.to(device) if class_w is not None else None,
                                 reduction="none")
            loss = (ce * w).mean()
            if training:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += float(loss.detach()) * len(y)
            all_pred.append(logits.argmax(1).cpu().numpy())
            all_true.append(y.cpu().numpy())
    return (total_loss / len(np.concatenate(all_true)),
            np.concatenate(all_true), np.concatenate(all_pred))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate(y_true, y_pred, id2label):
    labels = list(range(len(id2label)))
    names  = [id2label[i] for i in labels]
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro",
                                   labels=labels, zero_division=0)),
        "report":   classification_report(y_true, y_pred, labels=labels,
                                          target_names=names, zero_division=0,
                                          output_dict=True),
    }

## 10. Train

In [ ]:
best_f1, best_path = -1.0, os.path.join(OUT_DIR, "best_model.pt")

for epoch in range(1, EPOCHS + 1):
    tr_loss, _, _ = run_epoch(model, train_loader, device, optimizer, cw)
    _, yv, pv = run_epoch(model, val_loader, device, None, cw)
    vm = evaluate(yv, pv, id2label)
    print(f"[epoch {epoch}] train_loss={tr_loss:.4f}  "
          f"val_acc={vm['accuracy']:.4f}  val_macroF1={vm['macro_f1']:.4f}")
    if vm["macro_f1"] > best_f1:
        best_f1 = vm["macro_f1"]
        torch.save(model.state_dict(), best_path)
        print(f"           ^ new best -> {best_path}")

## 11. Test with best checkpoint

In [ ]:
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))

_, yt, pt = run_epoch(model, test_loader, device, None, cw)
tm = evaluate(yt, pt, id2label)
print(f"[TEST] acc={tm['accuracy']:.4f}  macroF1={tm['macro_f1']:.4f}\n")
for name in KEEP_LABELS:
    r = tm["report"].get(name, {})
    print(f"   {name:10s} P={r.get('precision',0):.3f} "
          f"R={r.get('recall',0):.3f} F1={r.get('f1-score',0):.3f} "
          f"n={int(r.get('support',0))}")

## 12. Save artifacts

In [ ]:
json.dump({"label2id": label2id, "id2label": id2label},
          open(os.path.join(OUT_DIR, "label_encoder.json"), "w"), indent=2)
json.dump({"val_best_macro_f1": best_f1, "test": tm},
          open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2)

te_rows = pairs.iloc[test_idx].reset_index(drop=True).copy()
te_rows["pred_label"] = [id2label[p] for p in pt]
te_rows[["item_i", "item_j", "weak_label", "pred_label", "confidence"]] \
    .to_parquet(os.path.join(OUT_DIR, "test_predictions.parquet"), index=False)

print("Saved to", OUT_DIR + "/")
print("  best_model.pt, label_encoder.json, metrics.json, test_predictions.parquet")

---
### If titles look useful, get real ones

`df_products.parquet` was built from KG artifacts, so `title` is a placeholder
(`item_<asin>`). To use real titles: add this cell to your KG notebook
**before** `candidates` is built and re-run this notebook:

```python
df[["item_id","title","brand","leaf_category"]] \\
    .reset_index(drop=True).to_parquet("df_products.parquet", index=False)
```

Brand and leaf_category are already correct — those come out of the KG.